# SEO Query Bucket Map

**Purpose:** Build a reusable mapping from raw Google Search Console queries to SEO intent buckets that mirror our paid search categories.

**Source table:** `lakehouse_production.common.gsc_search_analytics_d_1`

**Approach:**
1. **Rule-based bucketing (SQL)** — deterministic CASE statement applied to all distinct queries. Covers the vast majority of volume.
2. **AI classification (Python)** — sends remaining `Other` queries to OpenAI for classification. Catches long-tail and ambiguous queries the rules miss.
3. **Merge** — combines both into a single `seo_query_bucket_map` temp view for downstream use.

**Bucket priority order** (first match wins in the CASE):
1. Spanish — detected before English rules to avoid partial matches
2. Brand — our own properties (highest English priority)
3. Supplier — competitor REP brands
4. Aggregator — Power to Choose / PUCT
5. No Deposit
6. Price Sensitive
7. Rates
8. Companies — generic company-type searches
9. Geo — location-specific
10. Generic — broad electricity/energy terms
11. Other — fallback, sent to AI

## Step 1: Rule-Based Bucketing (SQL)

Extracts every distinct normalized query from the GSC table for our four domains, then classifies each with a priority-ordered CASE statement.

**Validated against 568K live queries (2025-01-01 onward).** Rule-based coverage: ~62% of queries get a specific bucket; ~38% fall to Other (see notes below on GSC breadth).

Key design decisions:
- `LOWER(TRIM(query))` normalizes casing and whitespace once in the CTE
- RLIKE uses word boundaries (`\\b`) to avoid substring false positives
- Spanish is checked first because words like "energia" would otherwise match Generic
- Brand takes precedence so "choosetexaspower rates" → Brand, not Rates
- Supplier list includes ~60 REP brands plus common misspellings and URL forms
- `\\s*` handles spacing variants ("direct energy" vs "directenergy")
- Geo list covers ~80 Texas cities/metros

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW seo_query_bucket_rules AS

WITH distinct_queries AS (
  -- Deduplicate and normalize: one row per unique query string
  SELECT DISTINCT
    LOWER(TRIM(query)) AS query
  FROM lakehouse_production.common.gsc_search_analytics_d_1
  WHERE date >= '2025-01-01'
    AND domain IN (
      'choosetexaspower.org',
      'saveonenergy.com',
      'chooseenergy.com',
      'texaselectricrates.com'
    )
    AND query IS NOT NULL
    AND TRIM(query) != ''
)

SELECT
  query,

  CASE
    --------------------------------------------------------------------------
    -- 1. SPANISH: detect before English rules so 'luz', 'energia', etc.
    --    don't leak into Generic or Companies
    --------------------------------------------------------------------------
    WHEN query RLIKE '(?i)\\b(luz|electricidad|electrica|energi(a|á)|compa(ñ|n)ia de|servicio de|pagar|cuenta de luz|tarifa|proveedor|sin dep(o|ó)sito|cr(e|é)dito|factura|recibo|planes? de|barato|barata|baratos|baratas|mejor(es)?\\s+(precio|tarifa|plan)|cambiar de|prepago)\\b'
      THEN 'Spanish'

    --------------------------------------------------------------------------
    -- 2. BRAND: our own properties — highest English priority
    --    \\s* handles spacing variants (e.g. "choose texas power" vs "choosetexaspower")
    --------------------------------------------------------------------------
    WHEN query RLIKE '(?i)(choosetexaspower|choose\\s*texas\\s*power|ctxp|saveonenergy|save\\s*on\\s*energy|chooseenergy|choose\\s*energy|texaselectricrates|texas\\s*electric\\s*rates)'
      THEN 'Brand'

    --------------------------------------------------------------------------
    -- 3. SUPPLIER: competitor REP brand names + common misspellings/URL forms
    --    ~60 brands. Add new entrants with |new_name.
    --------------------------------------------------------------------------
    WHEN query RLIKE '(?i)\\b(txu|reliant|direct\\s*energ|green\\s*mountain|gexa|champion\\s*energy|cirro|ambit|frontier\\s*(utilit|energy)|pulse\\s*power|4\\s*change|dynowatt|express\\s*energy|discount\\s*power|pennywise|tara\\s*energy|just\\s*energy|mp2|tri\\s*eagle|constellation|shell\\s*energy|octopus\\s*energy|chariot\\s*energy|rhythm|payless\\s*power|first\\s*choice\\s*power|startex|endurance|tomorrow\\s*energy|griddy|spark\\s*energy|veteran\\s*energy|infuse\\s*energy|value\\s*power|our\\s*energy|think\\s*energy|summer\\s*energy|breeze\\s*energy|nrg\\b|centerpoint|oncor|aep\\s*texas|entrust|trieagle|encompass|flagship\\s*power|liberty\\s*power|peco|wtu|bulb\\s*energy|ovo\\s*energy|lumo|eversource|national\\s*grid|nationalgrid|duke\\s*energy|dominion|xcel|alliant|dte\\b|comed\\b|ameren|pepco|bgee?\\b|ppl\\s*electric|firstenergy|exelon|vistra|nrg\\s*energy)\\b'
      THEN 'Supplier'

    --------------------------------------------------------------------------
    -- 4. AGGREGATOR: Power to Choose / PUCT / government comparison sites
    --------------------------------------------------------------------------
    WHEN query RLIKE '(?i)(power\\s*to\\s*choose|powertochoose|power2choose|ptc\\s*texas|puct|powerchoose|apples\\s*to\\s*apples)'
      THEN 'Aggregator'

    --------------------------------------------------------------------------
    -- 5. NO DEPOSIT: explicit bucket for credit-sensitive searchers
    --------------------------------------------------------------------------
    WHEN query RLIKE '(?i)(no deposit|no credit check|prepaid electric|prepaid energy|prepaid light|bad credit|without deposit|no\\s+credito)'
      THEN 'No Deposit'

    --------------------------------------------------------------------------
    -- 6. PRICE SENSITIVE: "cheap" intent keywords
    --------------------------------------------------------------------------
    WHEN query RLIKE '(?i)\\b(cheap(est)?|lowest|affordable|low cost|budget|inexpensive|best price|lowest price|most affordable|free electricity|free energy|free nights|free weekends)\\b'
      THEN 'Price Sensitive'

    --------------------------------------------------------------------------
    -- 7. RATES: people searching for pricing information
    --------------------------------------------------------------------------
    WHEN query RLIKE '(?i)\\b(rates?|pricing|price per kwh|cost of (electricity|energy)|kwh (rate|cost|price)|electricity cost|energy cost|how much.*(electricity|energy|kwh|cost)|price to compare|average.*(bill|cost).*electri)\\b'
      THEN 'Rates'

    --------------------------------------------------------------------------
    -- 8. COMPANIES: searching for a type of company, not a specific name
    --    Catches "electrical" alongside "electric", plus common misspellings
    --------------------------------------------------------------------------
    WHEN query RLIKE '(?i)(electric(ity|al)?\\s*compan|light\\s*compan|energy\\s*compan|energy\\s*service|electric(ity|al)?\\s*provider|energy\\s*provider|power\\s*compan|electric(ity|al)?\\s*service|energy\\s*supplier|utilit(y|ies)\\s*(compan|provider)|engery\\s*compan|eletric\\s*compan|electrician\\s*compan)'
      THEN 'Companies'

    --------------------------------------------------------------------------
    -- 9. GEO: location-focused queries (state + ~80 TX cities/metros)
    --    Only fires if no higher-priority bucket matched first
    --------------------------------------------------------------------------
    WHEN query RLIKE '(?i)\\b(texas|houston|dallas|fort worth|san antonio|austin|corpus christi|el paso|arlington|plano|lubbock|laredo|irving|mcallen|amarillo|brownsville|killeen|midland|odessa|waco|abilene|beaumont|round rock|garland|carrollton|denton|richardson|tyler|college station|wichita falls|galveston|katy|frisco|spring|conroe|sugar land|temple|longview|pearland|pflugerville|allen|edinburg|league city|mission|bryan|pasadena|mesquite|mckinney|san marcos|new braunfels|cedar park|georgetown|leander|euless|duncanville|copperas cove|weatherford|harlingen|victoria|lufkin|nacogdoches|texarkana|del rio|eagle pass|hurst|bedford|mansfield|rowlett|desoto|cedar hill|cleburne|corsicana|rockwall|waxahachie|seguin|bastrop|kyle|buda|liberty hill)\\b'
      THEN 'Geo'

    --------------------------------------------------------------------------
    -- 10. GENERIC: broad electricity / energy / utility terms
    --     "electrical" added alongside "electric" to catch variants
    --------------------------------------------------------------------------
    WHEN query RLIKE '(?i)\\b(electricity|electric(al)?|energy|power|utility|utilities|light bill|kilowatt|kwh|watt|deregulat|switch.*(provider|plan|service)|energy plan|electricity plan|power plan|renewable|solar|wind energy|green energy|smart meter|grid|thermostat|generator|outage|blackout|brownout)\\b'
      THEN 'Generic'

    --------------------------------------------------------------------------
    -- 11. OTHER: nothing matched — candidates for AI classification
    --------------------------------------------------------------------------
    ELSE 'Other'

  END AS query_bucket,

  'rule' AS classification_method

FROM distinct_queries;

### Quick sanity check: bucket distribution

In [ ]:
%sql
SELECT
  query_bucket,
  COUNT(*) AS query_count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_of_total
FROM seo_query_bucket_rules
GROUP BY query_bucket
ORDER BY query_count DESC;

### Spot-check a few examples per bucket

In [ ]:
%sql
-- Sample 5 queries from each bucket to verify classification
SELECT *
FROM (
  SELECT
    query,
    query_bucket,
    ROW_NUMBER() OVER (PARTITION BY query_bucket ORDER BY query) AS rn
  FROM seo_query_bucket_rules
)
WHERE rn <= 5
ORDER BY query_bucket, rn;

### Note on the Other bucket (~38% of queries)

GSC captures **every query** that generated an impression for our domains, including tangential topics our pages incidentally rank for. The ~215K Other queries break down roughly as:

| Category | Est. count | Examples |
|----------|-----------|---------|
| Gas / natural gas / fuel | ~42K | "gas prices in georgia", "natural gas per therm" |
| Appliances / HVAC | ~31K | "space heater", "water heater", "thermostat" |
| Sensors / switches / hardware | ~11K | "motion sensor light switch", "circuit breaker" |
| Weather / climate | ~10K | "weather forecast", "carbon emissions" |
| Tesla / EV | ~6K | "tesla powerwall", "electric car charging" |
| Electricity-related (missed by rules) | ~5K | Rare patterns, non-TX locations |
| Truly random | ~110K | Phone numbers, URLs, unrelated topics |

**This is expected for GSC data** and differs from paid search where you only track keywords you bid on.

**Options:**
1. **Keep as-is** — the AI step will reclassify electricity-related queries; the rest stay as Other
2. **Pre-filter** — optionally exclude gas/HVAC/hardware noise from the source CTE (see next cell)
3. **AI only on relevant subset** — filter the AI step to exclude obvious non-electricity queries to save cost

In [ ]:
%sql
-- OPTIONAL: Pre-filter to exclude non-electricity queries before AI classification.
-- This reduces the Other bucket sent to OpenAI from ~215K to ~130K, saving ~$40 in API cost.
-- Uncomment and run this cell to create a filtered view; the AI step below will use it.

-- CREATE OR REPLACE TEMP VIEW seo_query_bucket_rules_filtered AS
-- SELECT * FROM seo_query_bucket_rules
-- WHERE query_bucket != 'Other'
--    OR (
--      -- Keep Other queries that are plausibly about electricity/energy services
--      query NOT RLIKE '(?i)(\\b(gas price|gasoline|natural gas|propane|gallon|diesel|fuel oil|heating oil|gas bill|gas rate|gas supplier|gas company)\\b)'
--      AND query NOT RLIKE '(?i)(\\b(space heater|water heater|furnace|hvac|air condition|insulation|duct|heat pump)\\b)'
--      AND query NOT RLIKE '(?i)(\\b(motion sensor|light switch|outlet|circuit breaker|wiring diagram|fuse box|dimmer|led bulb|lamp|fixture)\\b)'
--      AND query NOT RLIKE '(?i)(\\b(tesla|electric vehicle|ev charg|electric car)\\b)'
--    );

---
## Step 2: AI Classification for Unmatched Queries (Python)

Takes the `Other` bucket queries and sends them to OpenAI in batches for classification.

**Requirements:**
- An OpenAI API key (set as `OPENAI_API_KEY` env var or Databricks secret)
- `openai` Python package installed in the cluster

**Design notes:**
- Batches of 50 queries per API call to balance cost vs. latency
- Structured JSON output so we can parse reliably
- Falls back to `Other` if parsing fails for any query
- Results stored as a temp view `seo_query_bucket_ai` for the merge step

In [ ]:
%python
import os
import json
import time
import pandas as pd
from openai import OpenAI

# ── Configuration ─────────────────────────────────────────────────────────
# Option 1: env var (works locally or if set on the cluster)
# Option 2: Databricks secret — uncomment the line below instead
# OPENAI_API_KEY = dbutils.secrets.get(scope="your-scope", key="openai-api-key")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "Set OPENAI_API_KEY as env var or via dbutils.secrets"

client = OpenAI(api_key=OPENAI_API_KEY)

VALID_BUCKETS = [
    "Supplier", "Aggregator", "Brand", "Companies", "Generic",
    "Geo", "Price Sensitive", "Spanish", "No Deposit", "Rates", "Other",
]

BATCH_SIZE = 50
MODEL = "gpt-4o-mini"  # cost-effective for classification; swap to gpt-4o if needed

SYSTEM_PROMPT = """
You are an SEO analyst for a Texas energy marketplace. Classify each search
query into exactly ONE of these buckets:

- Supplier: searches for a specific energy brand (TXU, Reliant, Gexa, etc.)
- Aggregator: searches for Power to Choose or government comparison sites
- Brand: searches for our brands (choosetexaspower, saveonenergy, chooseenergy, texaselectricrates)
- Companies: searches for a type of company ("electric company", "light company")
- Generic: broad electricity/energy/utility terms without specific intent
- Geo: queries where the primary intent is a location (city or state)
- Price Sensitive: queries with cheap/affordable/lowest intent
- Spanish: queries in Spanish
- No Deposit: queries about no-deposit or bad-credit plans
- Rates: queries specifically about rates, pricing, or cost
- Other: truly does not fit any bucket above

Return ONLY a JSON array. Each element must have:
  {"query": "<original query>", "bucket": "<one of the bucket names above>"}

Do not add any text outside the JSON array.
""".strip()


# ── Load unmatched queries ────────────────────────────────────────────────
other_df = spark.sql("""
    SELECT query FROM seo_query_bucket_rules WHERE query_bucket = 'Other'
""").toPandas()

print(f"{len(other_df)} queries classified as 'Other' — sending to OpenAI")

if other_df.empty:
    # Nothing to classify — create an empty view so Step 3 still works
    spark.createDataFrame([], "query STRING, ai_bucket STRING").createOrReplaceTempView("seo_query_bucket_ai")
    print("No unmatched queries. Skipping AI classification.")
else:
    queries = other_df["query"].tolist()

    # ── Batch classify ────────────────────────────────────────────────────
    results = []

    for i in range(0, len(queries), BATCH_SIZE):
        batch = queries[i : i + BATCH_SIZE]
        user_msg = json.dumps(batch)

        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg},
                ],
                temperature=0,
                max_tokens=4096,
            )
            raw = response.choices[0].message.content.strip()
            # Strip markdown fences if the model wraps output in ```json ... ```
            if raw.startswith("```"):
                raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
            parsed = json.loads(raw)

            for item in parsed:
                bucket = item.get("bucket", "Other")
                if bucket not in VALID_BUCKETS:
                    bucket = "Other"
                results.append({"query": item["query"], "ai_bucket": bucket})

        except Exception as e:
            print(f"  Batch {i // BATCH_SIZE + 1} failed: {e}. Defaulting to 'Other'.")
            for q in batch:
                results.append({"query": q, "ai_bucket": "Other"})

        # Respect rate limits
        time.sleep(0.5)

        if (i // BATCH_SIZE + 1) % 5 == 0:
            print(f"  Processed {min(i + BATCH_SIZE, len(queries))}/{len(queries)} queries")

    ai_df = pd.DataFrame(results)
    print(f"\nAI classification complete. Distribution:")
    print(ai_df["ai_bucket"].value_counts().to_string())

    # Register as temp view for the merge step
    spark.createDataFrame(ai_df).createOrReplaceTempView("seo_query_bucket_ai")

---
## Step 3: Combine Into Final Bucket Map

Merges the rule-based classifications with AI classifications.

Logic:
- If the rule engine assigned a real bucket (anything except `Other`), keep it.
- If the rule engine said `Other`, use the AI bucket.
- If the AI also said `Other` (or wasn't classified), it stays `Other`.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW seo_query_bucket_map AS

SELECT
  r.query,

  -- Final bucket: prefer rule, fall back to AI
  CASE
    WHEN r.query_bucket != 'Other' THEN r.query_bucket
    WHEN a.ai_bucket IS NOT NULL   THEN a.ai_bucket
    ELSE 'Other'
  END AS query_bucket,

  -- Track where the classification came from
  CASE
    WHEN r.query_bucket != 'Other' THEN 'rule'
    WHEN a.ai_bucket IS NOT NULL   THEN 'ai'
    ELSE 'unclassified'
  END AS classification_method

FROM seo_query_bucket_rules r
LEFT JOIN seo_query_bucket_ai a
  ON r.query = a.query;

### Final distribution check

In [ ]:
%sql
SELECT
  query_bucket,
  classification_method,
  COUNT(*) AS query_count
FROM seo_query_bucket_map
GROUP BY query_bucket, classification_method
ORDER BY query_bucket, classification_method;

In [ ]:
%sql
-- Overall bucket sizes
SELECT
  query_bucket,
  COUNT(*) AS query_count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_of_total
FROM seo_query_bucket_map
GROUP BY query_bucket
ORDER BY query_count DESC;

---
## Assumptions & Notes

1. **Table schema:** The query assumes `lakehouse_production.common.gsc_search_analytics_d_1` has columns `query`, `date`, and `domain`. No other columns are referenced.

2. **Bucket priority:** The CASE statement is ordered so that more specific buckets (Spanish, Brand, Supplier) fire before broader ones (Geo, Generic). This means a query like `"txu energy rates in dallas"` maps to **Supplier**, not Rates or Geo. Adjust the order if business needs differ.

3. **Supplier list maintenance:** The REP brand list (~60 names) will need periodic updates as companies enter/exit the Texas market. The regex is structured so adding a name is a single `|new_name` append inside the existing group.

4. **Geo coverage:** Includes ~80 Texas cities/metros. Smaller cities will fall through to Generic or Other and may be caught by the AI pass.

5. **GSC breadth vs. paid search:** GSC captures **all queries** that generated impressions for our domains, not just keywords we target. This means ~38% of queries are tangential (gas prices, HVAC equipment, lighting fixtures, non-TX locations). This is fundamentally different from paid search where you only track keywords you bid on. The optional pre-filter cell can exclude the noisiest non-electricity categories.

6. **AI cost estimate:** At ~50 queries per batch using `gpt-4o-mini`:
   - Full Other (~215K queries): ~4,300 API calls ≈ **$40-60**
   - With pre-filter (~130K queries): ~2,600 API calls ≈ **$25-35**
   - Consider sampling if cost is a concern (e.g. classify 10K, extrapolate patterns into new rules)

7. **Temp views vs. tables:** Everything is created as `TEMP VIEW` for iterative development. Once validated, persist with:
   ```sql
   CREATE OR REPLACE TABLE your_schema.seo_query_bucket_map AS
   SELECT * FROM seo_query_bucket_map
   ```

8. **No session joins yet:** This notebook only builds the query-to-bucket mapping. Joining to sessions, impressions, clicks, or rankings is a separate downstream step.

9. **Validated results (568K queries, 2025-01-01 onward):**

| Bucket | Count | % |
|--------|-------|---|
| Generic | 203K | 35.7% |
| Rates | 42K | 7.4% |
| Geo | 37K | 6.6% |
| Companies | 26K | 4.6% |
| Supplier | 26K | 4.6% |
| Price Sensitive | 12K | 2.1% |
| Spanish | 3.7K | 0.6% |
| No Deposit | 1.3K | 0.2% |
| Aggregator | 862 | 0.2% |
| Brand | 611 | 0.1% |
| Other | 215K | 37.9% |

---
## Step 4: GSC Page-Day Bucket Mix

Bridges GSC query data to session-level data by aggregating GSC rows to the **page + date + device + bucket** grain.

**What this does:**
1. Joins every GSC row to `seo_query_bucket_map` on the normalized query
2. Normalizes the GSC `page` URL (strip hash fragments, query params, lowercase, trim trailing slash) so it can later join to session `first_page_url`
3. Maps GSC `device` (UPPERCASE) to session-style `device_type` (Title Case)
4. Aggregates clicks, impressions, and weighted-average position per page-day-device-bucket

**Output grain:** one row per `(landing_page, date, device_type, query_bucket)`

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW gsc_page_day_bucket_mix AS

WITH gsc_normalized AS (
  -- Normalize GSC fields so they align with session-level join keys:
  --   page  → strip hash fragments, query params, lowercase, trim trailing slash
  --   device → map UPPERCASE to Title Case to match session device_type
  SELECT
    g.date,

    -- URL normalization: lowercase, strip #hash, trim trailing slash
    -- RTRIM arg order in Databricks: RTRIM(trimChars, string)
    RTRIM('/',
      LOWER(
        CASE
          WHEN POSITION('#' IN g.page) > 0
            THEN LEFT(g.page, POSITION('#' IN g.page) - 1)
          ELSE g.page
        END
      )
    ) AS landing_page,

    -- Device normalization: GSC DESKTOP/MOBILE/TABLET → Desktop/Mobile/Tablet
    CASE g.device
      WHEN 'DESKTOP' THEN 'Desktop'
      WHEN 'MOBILE'  THEN 'Mobile'
      WHEN 'TABLET'  THEN 'Tablet'
      ELSE INITCAP(g.device)
    END AS device_type,

    g.domain,
    LOWER(TRIM(g.query)) AS query_normalized,
    g.clicks,
    g.impressions,
    g.position
  FROM lakehouse_production.common.gsc_search_analytics_d_1 g
  WHERE g.date >= '2025-01-01'
    AND g.domain IN (
      'choosetexaspower.org',
      'saveonenergy.com',
      'chooseenergy.com',
      'texaselectricrates.com'
    )
)

-- Aggregate to page + date + device + bucket grain
SELECT
  gn.date                                                AS _date,
  gn.landing_page,
  gn.device_type,
  gn.domain,
  COALESCE(bm.query_bucket, 'Unmapped')                 AS query_bucket,
  SUM(gn.clicks)                                         AS clicks,
  SUM(gn.impressions)                                    AS impressions,
  -- Impression-weighted average position
  ROUND(
    SUM(gn.position * gn.impressions)
    / NULLIF(SUM(gn.impressions), 0),
    1
  )                                                      AS avg_position
FROM gsc_normalized gn
LEFT JOIN seo_query_bucket_map bm
  ON gn.query_normalized = bm.query
GROUP BY
  gn.date,
  gn.landing_page,
  gn.device_type,
  gn.domain,
  COALESCE(bm.query_bucket, 'Unmapped');

### Sanity check: page-day bucket mix

In [ ]:
%sql
-- Bucket distribution weighted by clicks (this is the click-weighted view of SEO traffic)
SELECT
  query_bucket,
  SUM(clicks)       AS total_clicks,
  SUM(impressions)  AS total_impressions,
  ROUND(SUM(clicks) * 100.0 / NULLIF(SUM(SUM(clicks)) OVER (), 0), 1) AS pct_clicks,
  COUNT(DISTINCT landing_page) AS unique_pages
FROM gsc_page_day_bucket_mix
GROUP BY query_bucket
ORDER BY total_clicks DESC;

---
## Step 5: Dominant Bucket per Page-Day-Device

For each `(landing_page, _date, device_type)`, picks the **single dominant bucket** — the one with the most clicks that day. This produces a one-to-one mapping that can cleanly join to sessions without duplication.

Includes supporting metrics:
- `dominant_bucket_clicks` / `dominant_bucket_impressions` — from the winning bucket
- `total_page_clicks` / `total_page_impressions` — across all buckets for that page-day-device
- `dominant_click_share` — confidence: what fraction of that page's clicks came from the dominant bucket

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW gsc_page_day_dominant_bucket AS

WITH page_totals AS (
  -- Total clicks/impressions per page-day-device (across all buckets)
  SELECT
    _date,
    landing_page,
    device_type,
    SUM(clicks)      AS total_page_clicks,
    SUM(impressions)  AS total_page_impressions
  FROM gsc_page_day_bucket_mix
  GROUP BY _date, landing_page, device_type
),

ranked AS (
  -- Rank buckets within each page-day-device by clicks (desc)
  -- Ties broken by impressions, then bucket name for determinism
  SELECT
    m.*,
    ROW_NUMBER() OVER (
      PARTITION BY m._date, m.landing_page, m.device_type
      ORDER BY m.clicks DESC, m.impressions DESC, m.query_bucket
    ) AS rn
  FROM gsc_page_day_bucket_mix m
)

SELECT
  r._date,
  r.landing_page,
  r.device_type,
  r.domain,

  -- The dominant (highest-click) bucket for this page-day-device
  r.query_bucket                                          AS seo_query_bucket,
  r.clicks                                                AS dominant_bucket_clicks,
  r.impressions                                           AS dominant_bucket_impressions,
  r.avg_position                                          AS dominant_bucket_avg_position,

  -- Page-level totals for context
  pt.total_page_clicks,
  pt.total_page_impressions,

  -- Confidence: what share of this page's clicks came from the dominant bucket
  ROUND(
    r.clicks * 1.0 / NULLIF(pt.total_page_clicks, 0),
    4
  )                                                       AS dominant_click_share

FROM ranked r
JOIN page_totals pt
  ON  r._date        = pt._date
  AND r.landing_page  = pt.landing_page
  AND r.device_type   = pt.device_type
WHERE r.rn = 1;

### Sanity check: dominant bucket distribution

In [ ]:
%sql
-- How often is the dominant bucket assignment confident?
SELECT
  CASE
    WHEN dominant_click_share >= 0.8 THEN '80-100% (high confidence)'
    WHEN dominant_click_share >= 0.5 THEN '50-79% (moderate)'
    WHEN dominant_click_share >= 0.3 THEN '30-49% (mixed)'
    ELSE 'Under 30% (low confidence)'
  END AS confidence_tier,
  COUNT(*)         AS page_day_device_rows,
  SUM(total_page_clicks) AS total_clicks_in_tier
FROM gsc_page_day_dominant_bucket
GROUP BY 1
ORDER BY 1;

---
## Step 6: SEO Session Fact

Joins the dominant-bucket layer to the true session-level fact table `energy_prod.data_science.mp_session_level_query`.

**Join keys:**
- `_date` = `_date`
- `RTRIM(LOWER(first_page_url), '/')` = `landing_page` (normalized in Step 4)
- `device_type` = `device_type`

**Important:** This is a LEFT JOIN from sessions → GSC. Sessions that don't match a GSC page-day-device row (e.g. Direct traffic, pages GSC doesn't track) will have NULL SEO columns. Filter to `marketing_channel = 'Organic'` for meaningful analysis.

**Output:** Every column from `mp_session_level_query` plus:
- `seo_query_bucket` — the dominant SEO bucket for the page that day
- `dominant_click_share` — how confident the bucket assignment is
- `gsc_clicks`, `gsc_impressions`, `gsc_avg_position` — GSC context for the page

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW seo_session_fact AS

SELECT
  s.*,

  -- SEO bucket context (NULL for non-organic or unmatched pages)
  d.seo_query_bucket,
  d.dominant_click_share,
  d.dominant_bucket_clicks       AS gsc_clicks,
  d.dominant_bucket_impressions  AS gsc_impressions,
  d.dominant_bucket_avg_position AS gsc_avg_position,
  d.total_page_clicks            AS gsc_total_page_clicks,
  d.total_page_impressions       AS gsc_total_page_impressions

FROM energy_prod.data_science.mp_session_level_query s

LEFT JOIN gsc_page_day_dominant_bucket d
  ON  s._date       = d._date
  AND s.device_type  = d.device_type
  -- Normalize session URL to match the GSC normalization from Step 4:
  -- lowercase, trim trailing slash (RTRIM arg order: trimChars, string)
  AND RTRIM('/', LOWER(s.first_page_url)) = d.landing_page

WHERE s._date >= '2025-01-01';

### Join quality check

In [ ]:
%sql
-- How well does the GSC bucket join to organic sessions?
SELECT
  marketing_channel,
  COUNT(*)                                                          AS sessions,
  COUNT(seo_query_bucket)                                           AS sessions_with_bucket,
  ROUND(COUNT(seo_query_bucket) * 100.0 / COUNT(*), 1)             AS match_rate_pct
FROM seo_session_fact
WHERE _date >= '2025-03-01'
GROUP BY marketing_channel
ORDER BY sessions DESC;

---
## Example Analyses

These queries demonstrate what `seo_session_fact` unlocks. All filter to `marketing_channel = 'Organic'` since the SEO bucket is only meaningful for organic traffic.

In [ ]:
%sql
-- VC (Visit Conversion) by SEO query bucket
SELECT
  COALESCE(seo_query_bucket, 'No GSC Match') AS seo_query_bucket,
  SUM(sessions)                               AS sessions,
  SUM(cart_orders)                             AS cart_orders,
  SUM(phone_orders)                            AS phone_orders,
  SUM(cart_orders) + SUM(phone_orders)         AS total_orders,
  ROUND(
    (SUM(cart_orders) + SUM(phone_orders)) * 100.0
    / NULLIF(SUM(sessions), 0),
    2
  )                                            AS vc_pct,
  ROUND(
    SUM(cart_orders) * 100.0 / NULLIF(SUM(sessions), 0),
    2
  )                                            AS cart_vc_pct,
  ROUND(AVG(gsc_avg_position), 1)              AS avg_gsc_position
FROM seo_session_fact
WHERE marketing_channel = 'Organic'
  AND _date >= '2025-03-01'
GROUP BY COALESCE(seo_query_bucket, 'No GSC Match')
ORDER BY sessions DESC;

In [ ]:
%sql
-- Landing page type mix by SEO bucket
SELECT
  COALESCE(seo_query_bucket, 'No GSC Match') AS seo_query_bucket,
  landing_page_type,
  SUM(sessions)                               AS sessions,
  ROUND(
    (SUM(cart_orders) + SUM(phone_orders)) * 100.0
    / NULLIF(SUM(sessions), 0),
    2
  )                                            AS vc_pct
FROM seo_session_fact
WHERE marketing_channel = 'Organic'
  AND _date >= '2025-03-01'
GROUP BY COALESCE(seo_query_bucket, 'No GSC Match'), landing_page_type
HAVING SUM(sessions) >= 100
ORDER BY seo_query_bucket, sessions DESC;

In [ ]:
%sql
-- Funnel drop-off by SEO bucket (organic only, recent month)
SELECT
  COALESCE(seo_query_bucket, 'No GSC Match') AS seo_query_bucket,
  SUM(sessions)                               AS sessions,
  SUM(zip_entry)                              AS zip_entries,
  SUM(is_product_viewed)                      AS product_views,
  SUM(has_cart)                               AS carts,
  SUM(cart_orders) + SUM(phone_orders)        AS orders,
  ROUND(SUM(zip_entry)          * 100.0 / NULLIF(SUM(sessions), 0), 1) AS zip_entry_rate,
  ROUND(SUM(is_product_viewed)  * 100.0 / NULLIF(SUM(sessions), 0), 1) AS product_view_rate,
  ROUND(SUM(has_cart)           * 100.0 / NULLIF(SUM(sessions), 0), 1) AS cart_rate,
  ROUND((SUM(cart_orders) + SUM(phone_orders)) * 100.0 / NULLIF(SUM(sessions), 0), 2) AS vc_pct
FROM seo_session_fact
WHERE marketing_channel = 'Organic'
  AND _date >= '2025-03-01'
GROUP BY COALESCE(seo_query_bucket, 'No GSC Match')
HAVING SUM(sessions) >= 50
ORDER BY sessions DESC;

In [ ]:
%sql
-- GSC position vs conversion: do higher rankings drive better VC?
SELECT
  CASE
    WHEN gsc_avg_position BETWEEN 1 AND 3   THEN '1-3 (top)'
    WHEN gsc_avg_position BETWEEN 4 AND 10  THEN '4-10 (page 1)'
    WHEN gsc_avg_position BETWEEN 11 AND 20 THEN '11-20 (page 2)'
    WHEN gsc_avg_position > 20              THEN '20+ (deep)'
    ELSE 'No position'
  END AS position_bucket,
  SUM(sessions)                              AS sessions,
  ROUND(
    (SUM(cart_orders) + SUM(phone_orders)) * 100.0
    / NULLIF(SUM(sessions), 0),
    2
  )                                           AS vc_pct,
  ROUND(AVG(gsc_avg_position), 1)             AS avg_position
FROM seo_session_fact
WHERE marketing_channel = 'Organic'
  AND _date >= '2025-03-01'
  AND seo_query_bucket IS NOT NULL
GROUP BY 1
ORDER BY avg_position;

---
## Pipeline Assumptions & Notes (Steps 4-6)

**URL Normalization:**
- GSC `page` is normalized by: `RTRIM('/', LOWER(strip_hash(page)))` — strips `#fragment`, lowercases, removes trailing `/`
- Session `first_page_url` is normalized the same way on the join: `RTRIM('/', LOWER(first_page_url))`
- Note: Databricks `RTRIM` arg order is `RTRIM(trimChars, string)`, not `RTRIM(string, trimChars)`
- Query parameters (`?param=value`) are not stripped because the GSC samples inspected did not contain them. If query params appear in `first_page_url`, add a `SPLIT` or `REGEXP_REPLACE` step.

**Device Normalization:**
- GSC uses uppercase: `DESKTOP`, `MOBILE`, `TABLET`
- Sessions use title case: `Desktop`, `Mobile`, `Tablet`
- The mapping in Step 4 converts GSC → title case with an explicit CASE statement

**Dominant Bucket Logic:**
- Ties in clicks are broken by impressions (descending), then alphabetical bucket name
- If a page-day-device has zero clicks for all buckets (impressions only), the bucket with most impressions wins
- `dominant_click_share` is NULL when `total_page_clicks = 0`

**Join Behavior:**
- `seo_session_fact` LEFT JOINs sessions to GSC, so non-organic or unmatched sessions keep all their columns with NULL SEO fields
- The `_date >= '2025-01-01'` filter on sessions matches the GSC date range. Adjust both together if expanding.
- Organic sessions that don't match GSC (e.g., edge-case URLs, timing mismatches) will have `seo_query_bucket IS NULL`

**Not a 1:1 query-session attribution:**
This pipeline assigns the *page-level dominant SEO bucket* to each session. It does NOT claim that a specific user searched a specific query. The intent is modeled segment-level analysis, not deterministic attribution.

**To persist as tables (once validated):**
```sql
CREATE OR REPLACE TABLE your_schema.gsc_page_day_bucket_mix AS SELECT * FROM gsc_page_day_bucket_mix;
CREATE OR REPLACE TABLE your_schema.gsc_page_day_dominant_bucket AS SELECT * FROM gsc_page_day_dominant_bucket;
CREATE OR REPLACE TABLE your_schema.seo_session_fact AS SELECT * FROM seo_session_fact;
```